Here we check how the network influences the behaviour of the automaton. For rule $\phi^9_{488,464}$ we know that a regular lattice does not allow for splitting; rather it converges to zones with straight boundaries. A random network, on the other hand, will create oscillating patterns. Somewhere halfway, the network is allowed to create homogeneous outcomes.

Here we investigate when that happens.

In [ ]:
# standard preamble for the Notebooks I use
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

# compact saving of data
import h5py

from src.automata import LLNA
from src.simulation import *
from src.rules import binary_indices, return_equivalent_rule, get_nonequiv_rules, return_life_like_dict
from src.networks import create_2d_torus_lattice, watts_strogatz_rewire
from src.analysis import median_and_percentiles_over_ensemble, mean_field_dens_propagation, derrida_map_analytical, hamming_weight, boolean_sens, mean_field_slope

%load_ext autoreload
%autoreload 2

In [ ]:
resolution = 9
beta, sigma = (488, 464)
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

# this rule is equivalent to itself
beta_equiv, sigma_equiv = return_equivalent_rule(resolution, B_set, S_set, return_decimals=True)
if (beta, sigma) == (beta_equiv, sigma_equiv):
    print("This rule is equivalent to itself.")

model = LLNA(resolution, B_set, S_set, iso=True)
model.diagram(degree=8, plot_dist=True)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 4))
fontsize=18

rho0_array = np.linspace(0,1,101)
delta0_array = np.linspace(0,1,101)
degrees= np.arange(1,21,1, dtype=int)

rho1_arrays = np.array([mean_field_dens_propagation(resolution, B_set, S_set, rho0_array, degree) for degree in degrees])
delta1_arrays = np.array([derrida_map_analytical(resolution, B_set, S_set, delta0_array, degree) for degree in degrees])

for degree, rho1_array, degree1_array in zip(degrees, rho1_arrays, delta1_arrays):
    axs[0].set_title("Mean field curve")
    axs[0].plot(rho0_array, rho1_array, label=f"Degree {degree}", color='k', alpha=0.5)
    axs[1].set_title("Derrida map")
    axs[1].plot(delta0_array, degree1_array, label=f"Degree {degree}", color='r', alpha=0.5)
for ax in axs:
    ax.plot([0,1], [0,1], lw=1, ls='--', color='k')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect(1)

plt.suptitle(f"Rule $\\phi^{resolution}_{{{beta},{sigma}}}$, degrees ${degrees[0]}$ to ${degrees[-1]}$", fontsize=fontsize)

In [ ]:
# configs_array[:,-1][0]

np.mean(configs_array[:,:,-1][0], axis=(1,2))

# (graphs, samples, time, width, height)

In [ ]:
# animations
import matplotlib.animation as animation
from IPython.display import HTML
from matplotlib import rcParams
rcParams['animation.embed_limit'] = 256  # Set a higher limit for embedding animations

def make_animation_object(grids, title=None):
    # Set up figure
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(grids[0])
    # Update function for animation
    def update(frame):
        im.set_array(grids[frame])
        ax.set_title(title, fontsize=20)
        ax.set_xticks([]); ax.set_yticks([])
        return [im]
    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=T+1, interval=100)
    plt.close()
    return ani

def make_animation_histogram(configs_list, rewiring_values, final_t, bins=np.linspace(0, 1, 12)):
    # Precompute maximum histogram height across all frames
    all_hist_vals = []
    for frame in range(len(configs_list)):
        frame_data = np.mean(configs_list[frame], axis=(1, 2))
        hist_vals, _ = np.histogram(frame_data, bins=bins, density=True)
        all_hist_vals.append(hist_vals)
    all_hist_vals = np.array(all_hist_vals)
    max_hist_val = np.max(all_hist_vals)

    # Set up figure
    fig, ax = plt.subplots(figsize=(5, 5))
    fontsize=20

    final_configs = np.array([config[:,-1] for config in configs_list])

    # Initial histogram from first frame
    frame_data = np.mean(final_configs[0], axis=(1, 2))
    hist_vals, _ = np.histogram(frame_data, bins=bins, density=True)
    bar_container = ax.bar(bins[:-1], hist_vals, width=np.diff(bins), align='edge', edgecolor='black')

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, max_hist_val * 1.05)
    ax.set_xlabel('Value', fontsize=fontsize)
    ax.set_ylabel('Density', fontsize=fontsize)
    ax.set_title(f"Global density at $t={final_t}$\nRewiring probability: {np.round(rewiring_values[0],2)}", fontsize=fontsize)

    # Update function
    def update(frame):
        frame_data = np.mean(final_configs[frame], axis=(1, 2))
        hist_vals, _ = np.histogram(frame_data, bins=bins, density=True)

        for rect, h in zip(bar_container, hist_vals):
            rect.set_height(h)
        ax.set_title(f"Global density at $t={final_t}$\nRewiring probability: {np.round(rewiring_values[frame],2)}", fontsize=fontsize)
        return bar_container

    ani = animation.FuncAnimation(
        fig, update, frames=len(configs_list), interval=100, blit=False
    )
    plt.close()
    return ani

# Make various small-world graphs for different rewiring probabilities

### 1. Make many Watts-Strogatz graphs with log-spaced rewiring probability

In [ ]:
# Parameters
width = 50                  # Grid size (L x L) WATCH OUT with values, becomes intensive fast. 200 takes 10 minutes
num_nodes = width**2
num_graphs = 49             # Number of graphs to generate
rewiring_prob_array = np.logspace(-2,0,num_graphs)         # Rewiring probability
degree = 8
num_edges = num_nodes*degree//2

# Create toroidal lattice and rewire.
# REWIRING CAN TAKE A WHILE! especially for large rewiring probabilities
lattice_graph = create_2d_torus_lattice(width, degree=degree)
watts_strogatz_graph_array = np.array([watts_strogatz_rewire(lattice_graph, rewiring_prob) for rewiring_prob in tqdm(rewiring_prob_array, total=num_graphs)])

# find edges
watts_strogatz_edges_list = []
for watts_strogatz_graph in watts_strogatz_graph_array:
    watts_strogatz_graph.to_directed()
    watts_strogatz_edges = tc.tensor(watts_strogatz_graph.get_edgelist()).T
    watts_strogatz_edges_list.append(watts_strogatz_edges)
    watts_strogatz_graph.to_undirected()
# watts_strogatz_edges_array = np.array(watts_strogatz_edges_list)

### 2. Make multiple runs on these graphs

In [ ]:
def init_config_with_dens(N, dens):
    # defines a random initial configuration with a fixed state density
    s0 = np.zeros(N, dtype=int)
    s0[:np.round(dens*N).astype(int)] = 1.
    np.random.shuffle(s0)
    return s0

T = 200
num_config = 200
init_dens = 0.5

configs_list = []
for watts_strogatz_edges in tqdm(watts_strogatz_edges_list, total=num_graphs):
    # run the network automaton
    init_configs = np.array([init_config_with_dens(num_nodes, init_dens) for _ in range(num_config)])
    configs = model.forward(watts_strogatz_edges, tc.tensor(init_configs), T=T).numpy().astype(int)
    # put back in L x L shape 
    configs = configs.reshape((num_config, T+1, width, width))
    configs_list.append(configs)
# configs_ = np.array(configs_list)

In [ ]:
graph_idx = 5
sample_idx = 7
ani = make_animation_object(configs_list[graph_idx][sample_idx], title=f"{model.__str__(latex=True)} on W-S graph ($p={rewiring_prob_array[graph_idx]:.2f}$)")
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

### 3. Make animation of final-state histograms

In [ ]:
# original animation
ani = make_animation_histogram(configs_list, rewiring_prob_array, T)
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

In [ ]:
# save some GIFs for future reference

SAVEGIFS = False
life_like_name = '488-464'

if SAVEGIFS:
    savename = f"animation_watts-strogatz_various-rewirings_{life_like_name}_{width}x{width}_T{T}.gif"
    # Save animations as GIFs
    loc = '../figures/gifs/'
    ani.save(loc+savename, writer='pillow', fps=8)  # Using Pillow
    # show inline
    # HTML(ani.to_jshtml())

The sweet spot (for this time duration) seems to be between 0.13 and 0.22.

# 4. Plot the variance over subsequent graphs

In [ ]:
final_configs = np.array([config[:,-1] for config in configs_list])
final_dens = np.mean(final_configs, axis=(2,3))
std_configs = np.std(final_dens, axis=1)

fontsize=20

plt.plot(rewiring_prob_array, std_configs)
plt.plot([.5,.5], [np.min(std_configs),np.max(std_configs)], 'k:')
plt.xscale('log')
plt.xlabel("Rewiring probability", fontsize=fontsize)
plt.ylabel("Standard deviation of\nfinal-state densities", fontsize=fontsize)

This shows the same information as the animation, but static. We expect that the curve will rise for intermediate values of the rewiring probability, if we wait longer.

TO DO: check the time until convergence for various rewiring probabilities (with a cut-off value cause we cannot wait indefinitely)